
# Batch Gradient Descent — A Full, Interactive Walkthrough

*From the math, to your from-scratch code, to a real 442-patient clinical dataset.*

This notebook rebuilds `Batch_Gradient_descent.ipynb` from the ground up and adds everything
around it: the calculus behind every line, an interactive 3D loss surface you can rotate,
convergence comparisons at different learning rates, a scaled-vs-unscaled experiment, a
from-scratch Stochastic/Mini-batch comparison, and a check against scikit-learn's
`LinearRegression` on the real diabetes dataset used in your annotated PDF.

**Contents**

1. [The big idea](#1.-The-big-idea-(no-math-yet))
2. [Every term, defined](#2.-Glossary)
3. [The math, derived step by step](#3.-The-math,-derived-step-by-step)
4. [Loading the real dataset](#4.-Loading-the-real-dataset)
5. [Exploring the data (interactive)](#5.-Exploring-the-data-(interactive))
6. [Batch Gradient Descent — from scratch](#6.-Batch-Gradient-Descent-—-from-scratch)
7. [Watching it learn — interactive loss curve](#7.-Watching-it-learn-—-interactive-loss-curve)
8. [The loss surface in 3D — with the descent path drawn on it](#8.-The-loss-surface-in-3D)
9. [Checking our answer against scikit-learn](#9.-Checking-our-answer-against-scikit-learn)
10. [Why the learning rate matters — interactive comparison](#10.-Why-the-learning-rate-matters)
11. [Why feature scaling matters](#11.-Why-feature-scaling-matters)
12. [Batch vs. Stochastic vs. Mini-batch](#12.-Batch-vs.-Stochastic-vs.-Mini-batch)
13. [Summary & references](#13.-Summary-&-references)



## 1. The big idea (no math yet)

Imagine standing on a hillside in thick fog. You can't see the valley floor, but you can feel
which way the ground slopes under your feet. So you take a small step downhill, feel the slope
again, and step again. Eventually you reach the bottom — without ever seeing the whole
landscape.

That is gradient descent. Translated into machine learning terms:

| Fog-hill idea | ML term |
|---|---|
| The landscape | The **cost function** — how wrong the model's predictions are |
| Your position on the hill | The model's current **weights** (coefficients) and **bias** (intercept) |
| The slope under your feet | The **gradient** — the direction of steepest *increase* in error |
| A step downhill | One **parameter update** |
| Reaching the bottom | **Convergence** — the point where the model has learned |

**"Batch"** answers one specific question: *how much ground do you feel before taking a step?*
In batch gradient descent, you check the slope using **every single training example** before
moving even once. It's slow per step, but each step is a confident, stable average over the
whole dataset — no guessing from a handful of points.



## 2. Glossary

| Term | Plain-English meaning |
|---|---|
| **Cost / loss function** | One number scoring how wrong the model is on average. Lower = better. |
| **Gradient** | The direction (and steepness) of fastest *increase* in the cost. We move *against* it. |
| **Learning rate (α)** | How big a step to take downhill. Too small → painfully slow. Too large → overshoot or diverge. |
| **Epoch** | One full pass over the entire training set, ending in one parameter update. |
| **Weights / coefficients** | The numbers each input feature gets multiplied by. |
| **Intercept / bias** | The baseline prediction when every feature is zero. |
| **Convergence** | The point where further steps stop meaningfully lowering the cost. |
| **Feature scaling** | Rescaling inputs to comparable ranges, so one learning rate suits every feature. |



## 3. The math, derived step by step

### 3.1 The model

For a row of features $x = (x_1, x_2, \dots, x_d)$, the linear model predicts

$$\hat{y} = w_1 x_1 + w_2 x_2 + \cdots + w_d x_d + b = x \cdot w + b$$

Stacking every training row into a matrix $X$ (shape $n \times d$), all predictions at once are

$$\hat{y} = Xw + b$$

### 3.2 The cost function

We score the whole batch of predictions with **Mean Squared Error**:

$$J(w,b) = \frac{1}{n}\sum_{i=1}^{n}\left(y_i - \hat{y}_i\right)^2$$

Squaring makes every error positive and penalizes large misses more than small ones. $n$ is
the *entire* training set — this is what makes it "batch."

### 3.3 Deriving the gradient with respect to the intercept $b$

Substitute $\hat{y}_i = x_i \cdot w + b$ and differentiate $J$ with respect to $b$ using the
chain rule:

$$
\frac{\partial J}{\partial b}
= \frac{1}{n}\sum_{i=1}^n \frac{\partial}{\partial b}\left(y_i - \hat{y}_i\right)^2
= \frac{1}{n}\sum_{i=1}^n 2\left(y_i - \hat{y}_i\right)\cdot\frac{\partial (y_i-\hat{y}_i)}{\partial b}
$$

Since $\hat{y}_i$ contains $+b$, we have $\dfrac{\partial(y_i - \hat{y}_i)}{\partial b} = -1$, so

$$\frac{\partial J}{\partial b} = -\frac{2}{n}\sum_{i=1}^n \left(y_i - \hat{y}_i\right) = -2\cdot\text{mean}(y-\hat{y})$$

**This is exactly** `interceptder = -2 * np.mean(y_train - y_hat)` in the from-scratch class below.

### 3.4 Deriving the gradient with respect to the weights $w$

The same chain rule, but now $\dfrac{\partial(y_i-\hat{y}_i)}{\partial w_j} = -x_{ij}$:

$$
\frac{\partial J}{\partial w_j} = \frac{1}{n}\sum_{i=1}^n 2\left(y_i-\hat{y}_i\right)\cdot(-x_{ij})
= -\frac{2}{n}\sum_{i=1}^n x_{ij}\left(y_i - \hat{y}_i\right)
$$

Written for the whole vector $w$ at once, using $X^{T}$ to weight every feature column by every
residual:

$$\nabla_w J = -\frac{2}{n} X^{T}\left(y - \hat{y}\right)$$

**This is exactly** `coef_der = -2 * np.dot(x_train.T, (y_train - y_hat)) / x_train.shape[0]`.

### 3.5 The update rule

Move a small step of size $\alpha$ (the learning rate) *opposite* the gradient, because the
gradient points uphill and we want to go down:

$$w \leftarrow w - \alpha\,\nabla_w J \qquad\qquad b \leftarrow b - \alpha\,\frac{\partial J}{\partial b}$$

Repeating this for a fixed number of epochs is the entire algorithm.


In [1]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
np.random.seed(42)



## 4. Loading the real dataset

This is the same **diabetes dataset** your PDF ran scikit-learn's `LinearRegression` on: ten
baseline measurements for 442 real diabetes patients, plus a quantitative measure of disease
progression one year later (Efron, Hastie, Johnstone & Tibshirani, 2004 — *Least Angle
Regression*, Annals of Statistics). Here we load the **raw, un-standardized** values so the
units are interpretable (years, mg/dL, mmHg, etc.), then scale them ourselves later — on
purpose, so the effect of scaling is something we control and can see.


In [2]:

diabetes = load_diabetes(scaled=False, as_frame=True)
df = diabetes.frame.rename(columns={'target': 'disease_progression'})
print(diabetes.DESCR[:620])
df.head(10)


.. _diabetes_dataset:

Diabetes dataset
----------------

Ten baseline variables, age, sex, body mass index, average blood
pressure, and six blood serum measurements were obtained for each of n =
442 diabetes patients, as well as the response of interest, a
quantitative measure of disease progression one year after baseline.

**Data Set Characteristics:**

:Number of Instances: 442

:Number of Attributes: First 10 columns are numeric predictive values

:Target: Column 11 is a quantitative measure of disease progression one year after baseline

:Attribute Information:
    - age     age in years
    - sex
    - bmi


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,disease_progression
0,59.0,2.0,32.1,101.0,157.0,93.2,38.0,4.00,4.8598,87.0,151.0
1,48.0,1.0,21.6,87.0,183.0,103.2,70.0,3.00,3.8918,69.0,75.0
2,72.0,2.0,30.5,93.0,156.0,93.6,41.0,4.00,4.6728,85.0,141.0
3,24.0,1.0,25.3,84.0,198.0,131.4,40.0,5.00,4.8903,89.0,206.0
4,50.0,1.0,23.0,101.0,192.0,125.4,52.0,4.00,4.2905,80.0,135.0
5,23.0,1.0,22.6,89.0,139.0,64.8,61.0,2.00,4.1897,68.0,97.0
6,36.0,2.0,22.0,90.0,160.0,99.6,50.0,3.00,3.9512,82.0,138.0
7,66.0,2.0,26.2,114.0,255.0,185.0,56.0,4.55,4.2485,92.0,63.0
8,60.0,2.0,32.1,83.0,179.0,119.4,42.0,4.00,4.4773,94.0,110.0
9,29.0,1.0,30.0,85.0,180.0,93.4,43.0,4.00,5.3845,88.0,310.0


In [3]:

df.describe().T[['mean','std','min','max']]


,mean,std,min,max
age,48.518100,13.109028,19.0000,79.000
sex,1.468326,0.499561,1.0000,2.000
bmi,26.375792,4.418122,18.0000,42.200
bp,94.647014,13.831283,62.0000,133.000
s1,189.140271,34.608052,97.0000,301.000
s2,115.439140,30.413081,41.6000,242.400
s3,49.788462,12.934202,22.0000,99.000
s4,4.070249,1.290450,2.0000,9.090
s5,4.641411,0.522391,3.2581,6.107
s6,91.260181,11.496335,58.0000,124.000



## 5. Exploring the data (interactive)

Ten real, physiologically meaningful features — far more than the 2D or 3D scatter plots we
could eyeball by hand. This is exactly the kind of problem gradient descent is built for:
finding the best combination of many weights at once, algorithmically, rather than by
inspection.

Hover over the heatmap and scatter plot below — both are interactive Plotly figures.


In [4]:

corr = df.corr(numeric_only=True)
fig = px.imshow(
    corr, text_auto='.2f', color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
    title='Correlation between every feature and disease progression'
)
fig.update_layout(width=780, height=680)
fig.show()


In [5]:

fig = px.scatter(
    df, x='bmi', y='disease_progression', color='bp',
    color_continuous_scale='Viridis', trendline='ols',
    title='BMI vs. disease progression (color = blood pressure)',
    labels={'bmi':'Body mass index', 'disease_progression':'Disease progression (1yr)', 'bp':'Blood pressure'}
)
fig.update_layout(width=780, height=520)
fig.show()



## 6. Batch Gradient Descent — from scratch

a `loss_history_` list so we can *watch* it learn in the sections below. Every other line —
the initialization, the prediction, both gradient formulas, and the update rule.


In [6]:

class GradientDescent:

    def __init__(self, learningrate=0.01, epochs=100):
        self.coef_ = None
        self.intercept_ = None
        self.lr = learningrate
        self.epochs = epochs
        self.loss_history_ = []          # <-- new: records MSE every epoch, for plotting

    def fit(self, x_train, y_train):
        self.intercept_ = 0
        self.coef_ = np.ones(x_train.shape[1])
        self.loss_history_ = []

        for i in range(self.epochs):
            y_hat = np.dot(x_train, self.coef_) + self.intercept_

            error = y_train - y_hat
            self.loss_history_.append(np.mean(error ** 2))     # MSE this epoch

            interceptder = -2 * np.mean(error)
            self.intercept_ = self.intercept_ - (self.lr * interceptder)

            coef_der = -2 * np.dot(x_train.T, error) / x_train.shape[0]
            self.coef_ = self.coef_ - (self.lr * coef_der)

        print("intercept:", self.intercept_)
        print("coefficients:", self.coef_)

    def predict(self, x_test):
        return np.dot(x_test, self.coef_) + self.intercept_



We'll train on **four** of the ten real features — `age`, `bmi`, `bp`, and `s5` (log of serum
triglycerides) — so the loss surface later in this notebook can still be visualized in 3D by
holding two coefficients fixed. Features are standardized (z-scored) before training, and
we'll come back to *why* that matters in Section 11.


In [7]:

features = ['age', 'bmi', 'bp', 's5']
X = df[features].values
y = df['disease_progression'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = GradientDescent(learningrate=0.3, epochs=300)
model.fit(X_train_s, y_train)

y_pred = model.predict(X_test_s)
print(f"\nTest RMSE: {mean_squared_error(y_test, y_pred) ** 0.5:.2f}")
print(f"Test R²:   {r2_score(y_test, y_pred):.3f}")


intercept: 153.73654390934854
coefficients: [-1.12537926 30.72226901 14.01900058 23.69163456]

Test RMSE: 53.64
Test R²:   0.457



## 7. Watching it learn — interactive loss curve

Each point below is the Mean Squared Error measured **after one full batch update** — one
epoch. Hover to read exact values; the curve should fall steeply at first, then flatten out as
the model approaches the bottom of the valley (convergence).


In [8]:

fig = go.Figure()
fig.add_trace(go.Scatter(
    y=model.loss_history_, mode='lines', name='Training MSE',
    line=dict(color='#E3A857', width=2.5),
    fill='tozeroy', fillcolor='rgba(227,168,87,0.12)'
))
fig.update_layout(
    title='Batch Gradient Descent — loss vs. epoch',
    xaxis_title='Epoch', yaxis_title='Mean Squared Error',
    width=780, height=440, template='plotly_dark'
)
fig.show()



## 8. The loss surface in 3D

A model with 4 weights + 1 intercept lives in 5-dimensional parameter space — impossible to
plot directly. But we can **slice** that space: fix `age` and `s5` at the values gradient
descent converged to, and sweep only the `bmi` and `bp` coefficients across a grid. That gives
us an actual 3D bowl — the classic picture behind gradient descent — and we can overlay the
*real path* our optimizer took across all 300 epochs, projected onto this slice.

**Drag to rotate, scroll to zoom** — this is a live 3D Plotly figure.


In [9]:

# Rebuild training history for (bmi coef, bp coef) specifically, re-running fit manually
# so we can log the path of just those two coefficients alongside the other two held fixed.
lr, epochs = 0.3, 300
intercept = 0.0
coef = np.ones(X_train_s.shape[1])
path_bmi, path_bp, path_loss = [], [], []

for i in range(epochs):
    y_hat = np.dot(X_train_s, coef) + intercept
    error = y_train - y_hat
    path_bmi.append(coef[1]); path_bp.append(coef[2]); path_loss.append(np.mean(error**2))

    interceptder = -2 * np.mean(error)
    intercept -= lr * interceptder
    coef_der = -2 * np.dot(X_train_s.T, error) / X_train_s.shape[0]
    coef = coef - lr * coef_der

fixed_age, fixed_s5 = coef[0], coef[3]

def mse_for(bmi_coef, bp_coef):
    w = np.array([fixed_age, bmi_coef, bp_coef, fixed_s5])
    pred = np.dot(X_train_s, w) + intercept
    return np.mean((y_train - pred) ** 2)

grid_n = 45
bmi_range = np.linspace(min(path_bmi) - 15, max(path_bmi) + 15, grid_n)
bp_range  = np.linspace(min(path_bp) - 15, max(path_bp) + 15, grid_n)
Z = np.array([[mse_for(b, p) for b in bmi_range] for p in bp_range])

fig = go.Figure()
fig.add_trace(go.Surface(
    x=bmi_range, y=bp_range, z=Z, colorscale='Viridis', opacity=0.85,
    contours={"z": {"show": True, "usecolormap": True, "project_z": True}}
))
fig.add_trace(go.Scatter3d(
    x=path_bmi, y=path_bp, z=path_loss, mode='lines+markers',
    line=dict(color='#F1CD97', width=6),
    marker=dict(size=2.5, color='#F1CD97'),
    name='Gradient descent path'
))
fig.add_trace(go.Scatter3d(
    x=[path_bmi[-1]], y=[path_bp[-1]], z=[path_loss[-1]],
    mode='markers', marker=dict(size=7, color='#4FD1C5'), name='Final position'
))
fig.update_layout(
    title='MSE surface sliced across the bmi & bp coefficients, with the real descent path',
    scene=dict(xaxis_title='bmi coefficient', yaxis_title='bp coefficient', zaxis_title='MSE'),
    width=820, height=650, template='plotly_dark'
)
fig.show()



Notice how the path lands exactly in the bowl's lowest visible point — that *is* what
convergence looks like geometrically. Every epoch is one more bead on that gold trail, always
moving toward lower ground.



## 9. Checking our answer against scikit-learn

If batch gradient descent is implemented correctly, it should land close to the same answer as
scikit-learn's `LinearRegression`, which solves for the optimal weights directly (via the
normal equation) rather than iteratively. This is the same comparison your annotated PDF made —
here it's the from-scratch version against the library version, on the same standardized
features.


In [10]:

sk_model = LinearRegression()
sk_model.fit(X_train_s, y_train)

comparison = pd.DataFrame({
    'feature': features,
    'GradientDescent (from scratch)': model.coef_,
    'scikit-learn LinearRegression': sk_model.coef_
})
comparison


,feature,GradientDescent (from scratch),scikit-learn LinearRegression
0,age,-1.125379,-1.125379
1,bmi,30.722269,30.722269
2,bp,14.019001,14.019001
3,s5,23.691635,23.691635


In [11]:

fig = go.Figure()
fig.add_trace(go.Bar(x=features, y=model.coef_, name='From scratch (batch GD)', marker_color='#E3A857'))
fig.add_trace(go.Bar(x=features, y=sk_model.coef_, name='scikit-learn', marker_color='#4FD1C5'))
fig.update_layout(
    title='Learned coefficients: from-scratch batch GD vs. scikit-learn',
    barmode='group', width=780, height=440, template='plotly_dark',
    yaxis_title='Coefficient (standardized units)'
)
fig.show()

sk_pred = sk_model.predict(X_test_s)
print(f"scikit-learn  → RMSE {mean_squared_error(y_test, sk_pred)**0.5:.2f} | R² {r2_score(y_test, sk_pred):.3f}")
print(f"From scratch  → RMSE {mean_squared_error(y_test, y_pred)**0.5:.2f} | R² {r2_score(y_test, y_pred):.3f}")


scikit-learn  → RMSE 53.64 | R² 0.457
From scratch  → RMSE 53.64 | R² 0.457



## 10. Why the learning rate matters — interactive comparison

The learning rate α controls step size. Too small, and training crawls. Too large, and the
steps overshoot the valley and the loss explodes instead of shrinking. Below, the exact same
`GradientDescent` class is trained five times, changing only α. **Click a legend entry to
toggle that line on or off** in the interactive plot.


In [12]:

learning_rates = [0.001, 0.01, 0.1, 0.3, 0.8]
fig = go.Figure()

for lr_test in learning_rates:
    m = GradientDescent(learningrate=lr_test, epochs=150)
    m.fit(X_train_s, y_train)
    losses = np.clip(m.loss_history_, 0, 1e6)  # clip so a diverging run doesn't break the axis
    fig.add_trace(go.Scatter(y=losses, mode='lines', name=f'α = {lr_test}'))

fig.update_layout(
    title='Loss curves at five different learning rates (same data, same class, same epochs)',
    xaxis_title='Epoch', yaxis_title='MSE (clipped for display)',
    yaxis_type='log', width=780, height=460, template='plotly_dark'
)
fig.show()


intercept: 39.87991873304959
coefficients: [ 3.52009596 11.55146761  8.19842023 10.38286752]
intercept: 146.31168051880223
coefficients: [-0.11717489 29.43013877 14.62984766 23.47609791]
intercept: 153.73654390934811
coefficients: [-1.12537933 30.7222688  14.01900066 23.69163474]
intercept: 153.73654390934854
coefficients: [-1.12537926 30.72226901 14.01900058 23.69163456]
intercept: 1.2826266231562723e+39
coefficients: [-2.31883372e+53 -3.04182066e+53 -3.04832699e+53 -3.13340536e+53]



On a log-scaled y-axis: the smallest learning rate barely moves in 150 epochs, the largest
either overshoots badly or diverges outright, and somewhere in the middle sits a rate that
reaches the bottom quickly and stays there. There's no universal "correct" learning rate — it
depends on the scale of your data and features, which brings us to the next section.



## 11. Why feature scaling matters

Look back at the raw, un-standardized features: `age` ranges roughly 19–79 (years), while `s5`
ranges roughly 3.4–6.1 (a log-scaled lab value). A single learning rate has to work for
*every* coefficient simultaneously — but the gradient for a large-scale feature like `bp` will
naturally be much larger than for a small-scale feature like `s5`. One learning rate that's
sensible for one feature can be wildly unstable for another.

The cell below trains on the **raw, unscaled** features with a learning rate that works
perfectly well on the *scaled* version (Section 6) — the same `lr=0.3` — to demonstrate the
failure mode directly.


In [13]:

unscaled_model = GradientDescent(learningrate=0.3, epochs=50)
unscaled_model.fit(X_train, y_train)   # note: X_train, NOT X_train_s

fig = go.Figure()
fig.add_trace(go.Scatter(
    y=np.clip(unscaled_model.loss_history_, 0, 1e10), mode='lines',
    line=dict(color='#E27D5F', width=2.5), name='Unscaled features, lr=0.3'
))
fig.update_layout(
    title='Same learning rate, unscaled features → the loss explodes instead of shrinking',
    xaxis_title='Epoch', yaxis_title='MSE (clipped)', yaxis_type='log',
    width=780, height=440, template='plotly_dark'
)
fig.show()
print("First 5 epochs of MSE:", [round(v,1) for v in unscaled_model.loss_history_[:5]])


intercept: 5.3441356641343584e+190
coefficients: [2.67390353e+192 1.43278082e+192 5.17401354e+192 2.50428620e+191]


/tmp/ipykernel_131005/1352497296.py:19: RuntimeWarning: overflow encountered in square
  self.loss_history_.append(np.mean(error ** 2))     # MSE this epoch


First 5 epochs of MSE: [5282.0, 20684784279.5, 1.1406190627988626e+18, 6.289941637542408e+25, 3.4685871234110194e+33]



That divergence isn't a bug in the class — it's the direct, predictable consequence of mixing
unscaled features with a single global learning rate. `StandardScaler` (or scaling by hand with
`(x - mean) / std`) is what makes Section 6's run stable at the same learning rate.



## 12. Batch vs. Stochastic vs. Mini-batch

All three variants share the same update rule from Section 3.5 — they differ only in **how
much data informs each gradient calculation**.

| | Batch | Stochastic (SGD) | Mini-batch |
|---|---|---|---|
| Data per update | Entire training set | One random row | A small random subset |
| Path to the minimum | Smooth, stable | Noisy, zig-zagging | A middle ground |
| Speed per update | Slow | Fast | Fast |
| Typical use | Small/medium data, convex problems | Streaming/online learning | Deep learning (the modern default) |

Below, quick from-scratch implementations of all three train on the same data for a comparable
number of total gradient computations, so the *shape* of their convergence — not just the
final answer — is easy to compare.


In [14]:

def batch_gd(X, y, lr=0.3, epochs=150):
    n, d = X.shape
    b, w = 0.0, np.ones(d)
    hist = []
    for _ in range(epochs):
        yhat = X @ w + b
        err = y - yhat
        hist.append(np.mean(err**2))
        b -= lr * (-2 * np.mean(err))
        w -= lr * (-2 * X.T @ err / n)
    return w, b, hist

def stochastic_gd(X, y, lr=0.05, epochs=150):
    n, d = X.shape
    b, w = 0.0, np.ones(d)
    hist = []
    rng = np.random.default_rng(0)
    for _ in range(epochs):
        idx = rng.permutation(n)
        epoch_losses = []
        for i in idx:
            xi, yi = X[i], y[i]
            yhat = xi @ w + b
            err = yi - yhat
            epoch_losses.append(err**2)
            b -= lr * (-2 * err)
            w -= lr * (-2 * xi * err)
        hist.append(np.mean(epoch_losses))
    return w, b, hist

def minibatch_gd(X, y, lr=0.1, epochs=150, batch_size=16):
    n, d = X.shape
    b, w = 0.0, np.ones(d)
    hist = []
    rng = np.random.default_rng(0)
    for _ in range(epochs):
        idx = rng.permutation(n)
        epoch_losses = []
        for start in range(0, n, batch_size):
            batch_idx = idx[start:start+batch_size]
            xb, yb = X[batch_idx], y[batch_idx]
            yhat = xb @ w + b
            err = yb - yhat
            epoch_losses.append(np.mean(err**2))
            b -= lr * (-2 * np.mean(err))
            w -= lr * (-2 * xb.T @ err / len(batch_idx))
        hist.append(np.mean(epoch_losses))
    return w, b, hist

w_b, b_b, hist_b = batch_gd(X_train_s, y_train)
w_s, b_s, hist_s = stochastic_gd(X_train_s, y_train)
w_m, b_m, hist_m = minibatch_gd(X_train_s, y_train)

fig = go.Figure()
fig.add_trace(go.Scatter(y=hist_b, mode='lines', name='Batch', line=dict(color='#E3A857', width=2.5)))
fig.add_trace(go.Scatter(y=hist_s, mode='lines', name='Stochastic', line=dict(color='#E27D5F', width=1.5)))
fig.add_trace(go.Scatter(y=hist_m, mode='lines', name='Mini-batch', line=dict(color='#4FD1C5', width=2)))
fig.update_layout(
    title='Batch vs. Stochastic vs. Mini-batch — loss per epoch, same data',
    xaxis_title='Epoch', yaxis_title='MSE', yaxis_type='log',
    width=780, height=460, template='plotly_dark'
)
fig.show()



Batch traces the smoothest curve — every point is a true full-dataset average. Stochastic is
visibly noisier, since each of its updates reacts to a single row, though it can still reach a
similar final loss. Mini-batch sits between the two, which is exactly why it's the default
choice for training large neural networks: most of batch's stability, most of stochastic's
speed.



## 13. Summary & references

- **Batch gradient descent** computes the gradient of the cost function using the *entire*
  training set before making a single parameter update.
- The update rule — $w \leftarrow w - \alpha \nabla_w J$ — is the same rule whether you're
  training a two-parameter linear regression or a billion-parameter neural network.
- Correctness (Section 9), the learning rate (Section 10), and feature scaling (Section 11)
  are the three things most likely to make or break a from-scratch implementation like the one
  in your original notebook.

**References**

1. IBM — *What is Gradient Descent?* — https://www.ibm.com/think/topics/gradient-descent
2. Built In — *What Is Gradient Descent?* — https://builtin.com/data-science/gradient-descent
3. scikit-learn — *Toy datasets* (diabetes dataset description) — https://scikit-learn.org/stable/datasets/toy_dataset.html
4. Efron, Hastie, Johnstone & Tibshirani (2004) — *Least Angle Regression*, Annals of Statistics — https://web.stanford.edu/~hastie/Papers/LARS/LeastAngle_2002.pdf
5. scikit-learn — *Stochastic Gradient Descent* user guide — https://scikit-learn.org/stable/modules/sgd.html
